# 07 時間序列與預測 — 練習

用松柏護理之家退伍軍人症 line list 練習時間序列分析與短期預測。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
cases = df[df["infected"] == 1]

## 題目 1：建立每日住院數序列

1. 用 `hospitalization_date` 建立每日住院數時間序列
2. 補齊無住院的日期（`fill_value=0`）
3. 印出序列長度、日期範圍、住院總數
4. 畫出住院曲線（bar chart）+ 5 日滾動平均

In [ ]:
# TODO: 建立每日住院數序列
# TODO: 補齊無住院日
# TODO: 印出摘要
# TODO: 畫住院曲線 + 5 日滾動平均

## 題目 2：住院數預測與窗口比較

1. 用 3 日、5 日、7 日滾動平均預測每日住院數（記得 `shift(1)`）
2. 計算各窗口的 MAE
3. 哪個窗口表現最好？
4. 用最佳窗口畫 Actual vs Predicted 圖

In [ ]:
# TODO: 3 / 5 / 7 日窗口 MAE 比較
# TODO: 選最佳窗口
# TODO: 畫 Actual vs Predicted 圖

## 題目 3（挑戰題）：按嚴重度分組的流行曲線

1. 將病例按 `clinical_severity`（mild / moderate / severe）分組
2. 分別建立每日發病數序列
3. 用 **stacked bar chart** 畫出三條嚴重度的流行曲線
4. 觀察：重症病例是否集中在某個時間段？

In [ ]:
# TODO: 按嚴重度分組建立每日序列
# TODO: stacked bar chart
# TODO: 觀察與解讀

## 題目 4：腸病毒每週通報數的季節性型態（腸病毒情境）

1. 用 `report_date` 建立每日通報數序列（`asfreq("D", fill_value=0)`），再用 `resample("W").sum()` 彙整成**每週**通報數
2. 計算各月份的平均週通報數，找出通報數最高的月份
3. 用 `shift(1)` 建立「前一週」通報數，計算週對週變化量（本週 - 前一週）
4. 畫出每週流行曲線，標出通報數最高的幾週
5. 解讀：腸病毒的季節性型態呈現幾個高峰？可能與哪些兒童群聚活動有關？

In [ ]:
import numpy as np

# --- 資料：腸病毒（Enterovirus）2 年通報 line list ---
rng = np.random.default_rng(701)
start = pd.Timestamp("2024-01-01")
n_days = 730
dates_all = pd.date_range(start, periods=n_days, freq="D")
doy = np.array([d.dayofyear for d in dates_all]) % 366

# 兩個季節高峰：初夏（約 4 月）與開學季（約 9-10 月）
peak1 = np.exp(-0.5 * ((doy - 100) / 22) ** 2)
peak2 = np.exp(-0.5 * ((doy - 270) / 28) ** 2)
weights = 0.05 + peak1 + 0.7 * peak2
probs = weights / weights.sum()

n_cases = 480
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
    "age_group": rng.choice(["<5", "5-9", "10-14"], size=n_cases, p=[0.55, 0.30, 0.15]),
}).sort_values("report_date").reset_index(drop=True)

# TODO: 用 report_date 建立每日序列（asfreq），再 resample("W").sum() 彙整成每週通報數
# TODO: 計算各月份平均週通報數，找出高峰月份
# TODO: 用 shift(1) 計算週對週變化量
# TODO: 畫出每週流行曲線，標出通報數最高的幾週
# TODO: 解讀腸病毒的季節性型態與可能成因

## 題目 5：流感每日通報數的 SARIMA 季節性預測（流感情境）

1. 建立每日通報數時間序列（補齊缺日）
2. 用 `adfuller()` 檢定序列是否平穩
3. 切分訓練 / 測試集（最後 14 天當測試集）
4. 配適 `SARIMAX(order=(1,1,1), seasonal_order=(1,1,0,7))`，預測測試期間的每日通報數
5. 計算 SARIMA 的 MAE，並與「訓練集最後 7 日滾動平均」這個簡單基準比較
6. 解讀：SARIMA 是否比滾動平均基準更能捕捉「週末通報下降」的週期型態？

In [ ]:
import numpy as np

# --- 資料：流感（Influenza）120 天通報 line list ---
rng = np.random.default_rng(707)
start = pd.Timestamp("2025-11-01")
n_days = 120
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# 單一流感季：中段達到高峰（約第 60 天），加上週末通報偏低的型態
season_shape = np.exp(-0.5 * ((day_idx - 60) / 18) ** 2)
dow_weight = np.where(dates_all.dayofweek < 5, 1.0, 0.45)  # 假日就診/通報較少
weights = (0.08 + season_shape) * dow_weight
probs = weights / weights.sum()

n_cases = 560
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
}).sort_values("report_date").reset_index(drop=True)

# TODO: 建立每日通報數序列（asfreq("D", fill_value=0)）
# TODO: 用 adfuller() 檢定平穩性
# TODO: 切分訓練 / 測試集（最後 14 天為測試集）
# TODO: 配適 SARIMAX(order=(1,1,1), seasonal_order=(1,1,0,7))，預測測試期間
# TODO: 計算 SARIMA 的 MAE，並與訓練集最後 7 日滾動平均基準比較
# TODO: 解讀 SARIMA 是否有效捕捉週末通報下降的週期型態

## 題目 6：COVID-19 每日新增與滾動平均平滑（COVID-19 情境）

1. 用 `report_date` 建立每日新增病例數序列（補齊缺日）
2. 計算 3 日、7 日、14 日滾動平均，並以 `shift(1)` 當作「預測前一日觀測值」，計算各窗口的 MAE
3. 找出 MAE 最小的窗口
4. 分別找出「原始每日數」與「7 日滾動平均」兩者的高峰日，比較兩者是否一致
5. 解讀：為什麼原始每日新增數不適合直接拿來判讀流行曲線的高峰日？

In [ ]:
import numpy as np

# --- 資料：COVID-19 90 天通報 line list ---
rng = np.random.default_rng(719)
start = pd.Timestamp("2026-03-01")
n_days = 90
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# 單一波流行：先升後降，高峰約在第 40 天，並疊加「週末通報較少」的雜訊
wave_shape = np.exp(-0.5 * ((day_idx - 40) / 14) ** 2)
dow_weight = np.where(dates_all.dayofweek < 5, 1.0, 0.5)
weights = (0.05 + wave_shape) * dow_weight
probs = weights / weights.sum()

n_cases = 600
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
}).sort_values("report_date").reset_index(drop=True)

# TODO: 建立每日新增病例數序列（asfreq("D", fill_value=0)）
# TODO: 計算 3 / 7 / 14 日滾動平均，用 shift(1) 算 MAE 並找出最佳窗口
# TODO: 找出原始每日數與 7 日滾動平均各自的高峰日
# TODO: 畫出每日新增 bar chart + 7 日滾動平均線
# TODO: 解讀為何原始每日數不適合直接判讀高峰日

## 題目 7：登革熱夏季高峰的 Poisson / Negative Binomial 迴歸（登革熱情境）

1. 用 `report_date` 建立每日通報數序列（補齊缺日）
2. 建立 lag 特徵：`lag_1`（前一天）、`lag_2`（前兩天）、`day_idx`（天數趨勢）
3. 計算 dispersion ratio（variance / mean），判斷是否過度離散
4. 配適 Poisson 迴歸 `cases ~ lag_1 + lag_2 + day_idx`；若過度離散，改配適 Negative Binomial 迴歸並比較 AIC
5. 將 Negative Binomial 迴歸係數轉換為 IRR（`exp(coef)`），解讀 `lag_1` 的意義
6. 解讀：登革熱夏季高峰的可能成因，以及為什麼登革熱資料常出現過度離散

In [ ]:
import numpy as np

# --- 資料：登革熱（Dengue）一年通報 line list ---
rng = np.random.default_rng(709)
start = pd.Timestamp("2025-01-01")
n_days = 365
dates_all = pd.date_range(start, periods=n_days, freq="D")
doy = np.arange(n_days)

# 夏季高峰：約 6-9 月（病媒蚊密度隨氣溫濕度上升）
summer_shape = np.exp(-0.5 * ((doy - 210) / 35) ** 2)
weights = 0.05 + summer_shape
probs = weights / weights.sum()

n_cases = 520
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
    "township": rng.choice(["A區", "B區", "C區"], size=n_cases, p=[0.5, 0.3, 0.2]),
}).sort_values("report_date").reset_index(drop=True)

# TODO: 建立每日通報數序列（asfreq("D", fill_value=0)）
# TODO: 建立 lag_1、lag_2、day_idx 特徵（記得 dropna）
# TODO: 計算 dispersion ratio = variance / mean，判斷是否過度離散
# TODO: 配適 Poisson 迴歸，並視 dispersion 決定是否改用 Negative Binomial
# TODO: 將係數轉換為 IRR 並解讀 lag_1 的意義
# TODO: 解讀登革熱夏季高峰與過度離散的成因

## 題目 8：延續護理之家事件——熱水系統二次污染的預警挑戰（退伍軍人病情境，挑戰題）

松柏護理之家的退伍軍人症群聚事件落幕約一個月後，感控小組懷疑熱水系統的生物膜尚未清除乾淨，可能導致「二次污染、二次爆發」。這題延續主資料集的每日 onset 概念，模擬「第一波 outbreak ＋ 一段平靜期 ＋ 第二波 outbreak」的完整病程，測試你手上的預測工具在真正的考驗下表現如何。

1. 建立完整的每日發病數序列（含第一波、平靜期、第二波，缺日補 0）
2. **只用第一波與平靜期（前 51 天）的資料**訓練三個模型：
   - (a) 3 日滾動平均（取訓練集最後一個滾動平均值，當作未來每天的固定預測）
   - (b) Poisson 迴歸 + lag 特徵（`lag_1`、`lag_2`、`day_idx`），用迭代預測的方式往後推算
   - (c) ARIMA(1,1,1)
3. 用這三個模型「預測」第 52 天之後（涵蓋整個第二波）的每日發病數，計算各模型的 MAE
4. 畫出三模型預測 vs. 實際發病數的比較圖
5. 解讀：三個模型是否都低估了第二波？這對醫院感控與即時監測系統的啟示是什麼？

In [ ]:
import numpy as np

# --- 資料：松柏護理之家「二次污染」延伸情境（含第一波 + 平靜期 + 第二波）---
rng = np.random.default_rng(713)
start = pd.Timestamp("2026-02-01")

# 第一波：與主資料集類似的 21 天 outbreak，高峰約在第 8 天
wave1_days = 21
wave1_shape = np.exp(-0.5 * ((np.arange(wave1_days) - 8) / 3.5) ** 2)
wave1_dates = pd.date_range(start, periods=wave1_days, freq="D")

# 第二波：55 天後熱水系統再次污染，規模較小、較集中
wave2_start = start + pd.Timedelta(days=55)
wave2_days = 15
wave2_shape = np.exp(-0.5 * ((np.arange(wave2_days) - 6) / 3) ** 2)
wave2_dates = pd.date_range(wave2_start, periods=wave2_days, freq="D")

def _sample_wave(dates_wave, shape, n, rng):
    probs = shape / shape.sum()
    idx = rng.choice(len(dates_wave), size=n, p=probs)
    return dates_wave[idx]

onset1 = _sample_wave(wave1_dates, wave1_shape, 92, rng)
onset2 = _sample_wave(wave2_dates, wave2_shape, 34, rng)
onset_all = np.concatenate([onset1, onset2])

line_list = pd.DataFrame({
    "case_id": np.arange(1, len(onset_all) + 1),
    "symptom_onset_date": onset_all,
}).sort_values("symptom_onset_date").reset_index(drop=True)

# TODO: 用 symptom_onset_date 建立完整每日發病數序列（從第一波第一天到第二波最後一天，缺日補 0）
# TODO: 切出訓練集 = 前 51 天（第一波 + 平靜期），測試集 = 第 52 天之後（涵蓋第二波）
# TODO: 用訓練集配 (a) 3 日滾動平均固定值 (b) Poisson + lag 迭代預測 (c) ARIMA(1,1,1)，各自預測測試期間
# TODO: 計算三模型的 MAE，畫出預測 vs 實際比較圖
# TODO: 解讀三模型是否都低估第二波，並說明對即時監測與感控預警的啟示